# Improved Alzheimer's Detection Pipeline

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)


import joblib


In [3]:
diagnosis_df = pd.read_excel('CSI_7_MAL_2324_CW_resit_data.xlsx', sheet_name='Diagnosis target')
cognitive_df = pd.read_excel('CSI_7_MAL_2324_CW_resit_data.xlsx', sheet_name='Cognitive score')
brain_df = pd.read_excel('CSI_7_MAL_2324_CW_resit_data.xlsx', sheet_name='Data')

df = diagnosis_df.merge(cognitive_df, on=['RID','Test_data'], how='inner')
df = df.merge(brain_df, on=['RID','Test_data'], how='inner')

print(df.shape)
df.head()


(1737, 371)


,RID,Test_data,Diagnosis,FDG_PET,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,0,CN,1.36926,0.0,10.67,18.67,28,44.0,4.0,...,0.694,3835,1622,2.077,0.746,15683,4272,3.028,0.649,4396
1,3,0,AD,1.09079,4.5,22.00,31.00,20,22.0,1.0,...,0.591,3681,1734,1.942,0.696,10387,3316,2.545,0.686,3304
2,4,1,LMCI,NaN,1.0,14.33,21.33,27,37.0,7.0,...,0.588,4060,1728,2.18,0.607,11156,3598,2.67,0.631,1338
3,5,0,CN,1.29799,0.0,8.67,14.67,29,37.0,4.0,...,0.628,5180,1868,2.543,0.709,11579,3387,2.911,0.66,1623
4,6,0,LMCI,NaN,0.5,18.67,25.67,25,30.0,1.0,...,0.631,3078,1241,2.141,0.701,9641,2781,2.9,0.727,1035


In [4]:

TARGET = 'Diagnosis'

# Feature Engineering
if {'Hippocampus','WholeBrain'}.issubset(df.columns):
    df['hippo_brain_ratio'] = df['Hippocampus'] / (df['WholeBrain'] + 1e-6)

if {'Ventricles','WholeBrain'}.issubset(df.columns):
    df['ventricle_brain_ratio'] = df['Ventricles'] / (df['WholeBrain'] + 1e-6)

if {'Hippocampus','Entorhinal'}.issubset(df.columns):
    df['hippo_entorhinal_ratio'] = df['Hippocampus'] / (df['Entorhinal'] + 1e-6)

if {'Ventricles','Hippocampus'}.issubset(df.columns):
    df['brain_loss_index'] = df['Ventricles'] / (df['Hippocampus'] + 1e-6)

if {'ADAS13','MMSE'}.issubset(df.columns):
    df['adas_mmse_ratio'] = df['ADAS13'] / (df['MMSE'] + 1e-6)

if {'CDRSB','MMSE'}.issubset(df.columns):
    df['cdr_mmse_ratio'] = df['CDRSB'] / (df['MMSE'] + 1e-6)

if {'MMSE','ADAS13'}.issubset(df.columns):
    df['mmse_minus_adas'] = df['MMSE'] - df['ADAS13']

if {'Hippocampus','Ventricles'}.issubset(df.columns):
    df['hippo_vent_ratio'] = df['Hippocampus'] / (df['Ventricles'] + 1e-6)

if {'CDRSB','ADAS13'}.issubset(df.columns):
    df['cdr_adas_ratio'] = df['CDRSB'] / (df['ADAS13'] + 1e-6)

X = df.drop(columns=[TARGET, 'RID', 'Test_data'], errors='ignore')
y = df[TARGET]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


/var/folders/yl/6m_wzs1n27g7_jms470dmjfc0000gn/T/ipykernel_4542/1361743303.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hippo_brain_ratio'] = df['Hippocampus'] / (df['WholeBrain'] + 1e-6)
/var/folders/yl/6m_wzs1n27g7_jms470dmjfc0000gn/T/ipykernel_4542/1361743303.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ventricle_brain_ratio'] = df['Ventricles'] / (df['WholeBrain'] + 1e-6)
/var/folders/yl/6m_wzs1n27g7_jms470dmjfc0000gn/T/ipykernel_4542/1361743303.py:11: PerformanceWarning: DataFrame is highly fragmented.

In [5]:

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

for col in categorical_cols:
    X[col] = X[col].astype(str)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            SimpleImputer(strategy='median'),
            numeric_cols
        ),
        (
            'cat',
            SkPipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_cols
        )
    ]
)

In [6]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)


In [7]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from catboost import CatBoostClassifier

best_model = CatBoostClassifier(
    iterations=2000,
    depth=8,
    learning_rate=0.02,
    l2_leaf_reg=5,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_seed=42,
    verbose=False
)

In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('sampling', SMOTE(random_state=42)),
    ('selector', SelectKBest(mutual_info_classif, k=75)),
    ('classifier', best_model)
])

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1
)

print("CV Scores:", scores)
print("Mean F1:", scores.mean())
print("Std:", scores.std())

CV Scores: [0.79502613 0.78239643 0.78368781 0.7859568  0.78150826]
Mean F1: 0.7857150848866252
Std: 0.004890290342573819


In [14]:
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, pred))
print("Weighted F1:", f1_score(y_test, pred, average='weighted'))

auc = roc_auc_score(
    y_test,
    probs,
    multi_class='ovr'
)

print("ROC-AUC:", auc)

print("\nClassification Report")
print(classification_report(y_test, pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and multiclass values for target
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and multiclass values for target
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and multiclass values for target
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/cluster/_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and multiclass values fo

Accuracy: 0.7672413793103449
Balanced Accuracy: 0.7093823141588922
Weighted F1: 0.7638180747093853
ROC-AUC: 0.9545314659585659

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.93      0.88        68
           1       0.84      0.74      0.78        84
           2       0.68      0.87      0.76        62
           3       0.81      0.73      0.77       113
           4       0.35      0.29      0.32        21

    accuracy                           0.77       348
   macro avg       0.70      0.71      0.70       348
weighted avg       0.77      0.77      0.76       348


Confusion Matrix
[[63  0  1  4  0]
 [ 0 62  3  8 11]
 [ 1  0 54  7  0]
 [12  0 19 82  0]
 [ 0 12  3  0  6]]


In [ ]:
joblib.dump(
    pipeline,
    'alzheimers_catboost_optimized.pkl'
)

joblib.dump(
    label_encoder,
    'label_encoder.pkl'
)